# Modelo Bayesiano Poisson-Gamma para Eliminación Directa — World Cup 2026

El notebook `soccer_poisson_model.ipynb` original calcula un **λ fijo** por partido a partir de un perfil estático de cada selección (OV, DV, MV, FA) definido *antes* de que arrancara el torneo. Esa lambda nunca cambia, sin importar lo que pase en la cancha.

Para la eliminación directa (dieciseisavos, octavos, cuartos, semifinal, final) queremos lo contrario: que la tasa de goles de cada equipo se **actualice con cada partido real que se juega**. Para eso usamos un modelo **Poisson-Gamma** — el caso clásico de inferencia bayesiana conjugada, donde la actualización tiene fórmula cerrada (no hace falta MCMC ni simulación para actualizar).

**Decisión explícita de alcance:** este modelo se entrena *solo* con goles reales ya jugados (`resultados_reales.csv`, fase de grupos de los 32 equipos clasificados a dieciseisavos). No se mezcla con el perfil OV/DV/MV/FA del modelo anterior — esa fue una decisión deliberada para que la tasa de gol refleje lo que el equipo *demostró en la cancha*, no lo que se esperaba de él en el papel.

## 1. Librerías

In [1]:
import json
import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 40)

## 2. Carga de resultados reales (fase de grupos, 32 clasificados)

Cada fila es "un equipo en un partido": goles a favor y en contra desde su propia perspectiva. Por eso cada partido real aparece dos veces (una por cada equipo), salvo cuando el rival ya quedó eliminado en fase de grupos — en ese caso solo está la fila del equipo clasificado.

In [2]:
df = pd.read_csv('resultados_reales.csv', encoding='utf-8')
print(f'Filas (equipo-partido): {len(df)}')
print(f'Equipos distintos: {df["Equipo"].nunique()}')
df.head()

Filas (equipo-partido): 96
Equipos distintos: 32


,Fase,Grupo,Jornada,Equipo,Rival,Goles_Favor,Goles_Contra,Resultado
0,Grupos,A,1,México,Sudáfrica,2,0,V
1,Grupos,A,2,México,Corea del Sur,1,0,V
2,Grupos,A,3,México,República Checa,3,0,V
3,Grupos,A,1,Sudáfrica,México,0,2,D
4,Grupos,A,2,Sudáfrica,República Checa,1,1,E


## 3. Por qué Poisson-Gamma

**Verosimilitud (cómo se generan los goles):** asumimos, igual que el modelo original, que los goles de un equipo en un partido siguen una Poisson:

$$\text{goles}_k \mid \theta \sim \text{Poisson}(\theta)$$

**Prior:** en vez de fijar $\theta$ (la tasa de gol) como un número único, la tratamos como **incierta**, con una distribución Gamma:

$$\theta \sim \text{Gamma}(\alpha_0, \beta_0)$$

**Por qué Gamma y no otra cosa:** la Gamma es el *prior conjugado* de la Poisson — al observar nuevos goles, la posterior sigue siendo Gamma, con una actualización en forma cerrada:

$$\theta \mid \text{datos} \sim \text{Gamma}\Big(\alpha_0 + \sum_{k=1}^{n} \text{goles}_k,\ \ \beta_0 + n\Big)$$

Es decir: **cada partido nuevo suma sus goles a $\alpha$ y suma 1 a $\beta$.** No se reentrena nada, no hay optimización — es aritmética directa. Eso es exactamente lo que se necesita para que el modelo se vaya actualizando ronda por ronda (dieciseisavos → octavos → cuartos → ...) sin fricción: cuando un equipo juega su partido de octavos, ese resultado se suma a su historial y su tasa de gol posterior se recalcula al instante con la misma fórmula.

## 4. Dos tasas independientes por equipo: ataque y defensa

Para cada equipo $i$ modelamos **dos** parámetros Gamma-Poisson separados:

- $\text{ataque}_i$ — goles que anota por partido (a partir de `Goles_Favor`)
- $\text{defensa}_i$ — goles que recibe por partido (a partir de `Goles_Contra`)

**Decisión de modelado:** no ajustamos por la fuerza específica del rival de cada partido (eso requeriría un modelo conjunto tipo Dixon-Coles, resuelto por máxima verosimilitud iterativa — rompe la conjugación cerrada). Con 3 partidos de grupos por equipo, ese ajuste fino sería ruidoso de todas formas. Mantenemos el modelo simple: cada equipo tiene su propia tasa de anotar y de recibir, estimada con sus propios goles reales.

## 5. Hiperprior: Empirical Bayes con método de momentos

¿De dónde sale $\alpha_0, \beta_0$? No los elegimos a mano — los estimamos *de los mismos datos*, mirando cómo varía la tasa de gol entre los 32 equipos clasificados (esto es "Empirical Bayes": usamos la población de equipos para construir el prior que luego se aplica a cada equipo individual).

Para una tasa $\theta_i \sim \text{Gamma}(\alpha_0,\beta_0)$ con media $m=\alpha_0/\beta_0$ y varianza $v_0=\alpha_0/\beta_0^2$:

- La tasa observada por equipo es $\hat{\theta}_i = x_i / n_i$ (goles totales / partidos jugados).
- Su varianza muestral entre los 32 equipos, $\text{Var}(\hat{\theta}_i)$, mezcla **dos fuentes**: la varianza real entre equipos ($v_0$) y el ruido propio de la Poisson ($m/\bar{n}$, con $\bar n$=partidos promedio).

$$v_0 = \text{Var}(\hat\theta_i) - \frac{m}{\bar n} \qquad \beta_0 = \frac{m}{v_0} \qquad \alpha_0 = m\cdot\beta_0$$

Si $v_0 \le 0$ (la varianza observada es toda ruido, no hay diferencia real entre equipos), usamos un prior muy fuerte (alto $\beta_0$) — exactamente lo que vamos a encontrar para el ataque.

In [3]:
def empirical_bayes_gamma(goles_por_equipo: pd.Series, partidos_por_equipo: pd.Series):
    """Método de momentos: estima (a0, b0) de un prior Gamma a partir de tasas observadas por equipo."""
    tasas = goles_por_equipo / partidos_por_equipo
    m = tasas.mean()
    n_bar = partidos_por_equipo.mean()
    var_obs = tasas.var(ddof=1)
    v0 = var_obs - (m / n_bar)
    if v0 <= 0:
        # Sin señal real entre equipos: prior muy concentrado (fuerte shrinkage hacia la media global)
        v0 = m / n_bar * 0.05
    b0 = m / v0
    a0 = m * b0
    return a0, b0, m, var_obs, v0

agg = df.groupby('Equipo').agg(
    goles_favor=('Goles_Favor', 'sum'),
    goles_contra=('Goles_Contra', 'sum'),
    partidos=('Goles_Favor', 'count'),
)

a0_ataque, b0_ataque, m_ataque, var_ataque, v0_ataque = empirical_bayes_gamma(agg['goles_favor'], agg['partidos'])
a0_defensa, b0_defensa, m_defensa, var_defensa, v0_defensa = empirical_bayes_gamma(agg['goles_contra'], agg['partidos'])

print('--- ATAQUE ---')
print(f'media entre equipos (m): {m_ataque:.4f}  var. observada: {var_ataque:.4f}  var. previa (v0): {v0_ataque:.4f}')
print(f'a0 = {a0_ataque:.3f}   b0 = {b0_ataque:.3f}')
print()
print('--- DEFENSA ---')
print(f'media entre equipos (m): {m_defensa:.4f}  var. observada: {var_defensa:.4f}  var. previa (v0): {v0_defensa:.4f}')
print(f'a0 = {a0_defensa:.3f}   b0 = {b0_defensa:.3f}')

--- ATAQUE ---
media entre equipos (m): 1.8958  var. observada: 0.6483  var. previa (v0): 0.0164
a0 = 219.787   b0 = 115.932

--- DEFENSA ---
media entre equipos (m): 1.0521  var. observada: 0.4452  var. previa (v0): 0.0945
a0 = 11.709   b0 = 11.129


## 6. Hallazgo importante: poca diferenciación ofensiva entre clasificados

Los 32 equipos que llegaron a dieciseisavos ya son una muestra **pre-filtrada** de selecciones fuertes. Al medir la varianza de ataque entre ellos, resulta casi indistinguible del ruido puro de la Poisson ($v_0 \approx 0$) — es decir, con solo 3 partidos cada uno, **no hay evidencia fuerte de que un equipo ofensivamente sea mejor que otro** dentro de este grupo de 32. El prior de ataque queda muy concentrado ($\beta_0$ alto) y por lo tanto aplica **shrinkage agresivo**: todas las predicciones de ataque tienden a parecerse a la media global hasta que se acumulen más partidos reales de eliminación directa.

En **defensa** sí hay señal real por encima del ruido — algunos equipos claramente conceden menos que otros incluso dentro de este grupo selecto — así que el prior de defensa es más laxo y permite más diferenciación desde el inicio.

Esto no es un defecto del modelo: es lo que los datos realmente dicen. A medida que cada equipo juegue más partidos de knockout, su propia evidencia (no el prior poblacional) va a dominar la predicción.

## 7. Posterior inicial por equipo (con datos de grupos)

$$\mathbb{E}[\text{ataque}_i \mid \text{datos}] = \frac{\alpha_0 + \text{goles\_favor}_i}{\beta_0 + n_i} \qquad \mathbb{E}[\text{defensa}_i \mid \text{datos}] = \frac{\gamma_0 + \text{goles\_contra}_i}{\delta_0 + n_i}$$

In [4]:
def posterior_medias(agg: pd.DataFrame, a0_ataque, b0_ataque, a0_defensa, b0_defensa):
    out = agg.copy()
    out['ataque_post'] = (a0_ataque + out['goles_favor']) / (b0_ataque + out['partidos'])
    out['defensa_post'] = (a0_defensa + out['goles_contra']) / (b0_defensa + out['partidos'])
    return out

posteriors = posterior_medias(agg, a0_ataque, b0_ataque, a0_defensa, b0_defensa)
posteriors.sort_values('ataque_post', ascending=False)

,goles_favor,goles_contra,partidos,ataque_post,defensa_post
Equipo,,,,,
Alemania,10,4,3,1.932094,1.111800
Países Bajos,10,4,3,1.932094,1.111800
Francia,10,2,3,1.932094,0.970249
Canadá,8,3,3,1.915277,1.041025
Senegal,8,6,3,1.915277,1.253352
Noruega,8,7,3,1.915277,1.324127
Estados Unidos,8,4,3,1.915277,1.111800
Argentina,8,1,3,1.915277,0.899473
Suiza,7,3,3,1.906869,1.041025


## 8. Combinando ataque y defensa para predecir un partido

Para un partido entre el equipo $i$ y el equipo $j$, la lambda esperada de $i$ se obtiene cruzando *su* ataque con la defensa de $j$, normalizado por el promedio global de goles (para que un equipo "promedio" contra un rival "promedio" dé exactamente el promedio global):

$$\lambda_{i \to j} = \frac{\mathbb{E}[\text{ataque}_i] \times \mathbb{E}[\text{defensa}_j]}{\mu_{\text{global}}}$$

Usamos $\mu_{\text{global}} = m_{\text{ataque}}$ (la misma media global usada para construir el hiperprior), para que el modelo sea autoconsistente.

In [5]:
MU_GLOBAL = m_ataque
GOL_MAX = 7

def poisson_prob(lambd, n):
    return (math.exp(-lambd) * (lambd ** n)) / math.factorial(n)

def lambda_partido(equipo_a, equipo_b, posteriors, mu_global=MU_GLOBAL):
    fila_a, fila_b = posteriors.loc[equipo_a], posteriors.loc[equipo_b]
    lam_a = (fila_a['ataque_post'] * fila_b['defensa_post']) / mu_global
    lam_b = (fila_b['ataque_post'] * fila_a['defensa_post']) / mu_global
    return lam_a, lam_b

def analizar_partido_bayes(equipo_a, equipo_b, posteriors, mu_global=MU_GLOBAL, gol_max=GOL_MAX):
    lam_a, lam_b = lambda_partido(equipo_a, equipo_b, posteriors, mu_global)
    goles = range(gol_max + 1)
    pa = np.array([poisson_prob(lam_a, g) for g in goles])
    pb = np.array([poisson_prob(lam_b, g) for g in goles])
    mat = np.outer(pa, pb)
    p_a = float(np.sum(np.tril(mat, k=-1)))
    p_e = float(np.sum(np.diag(mat)))
    p_b = float(np.sum(np.triu(mat, k=1)))
    ganador = equipo_a if p_a == max(p_a, p_e, p_b) else (equipo_b if p_b == max(p_a, p_e, p_b) else 'Empate')
    return {
        'Equipo_A': equipo_a, 'Equipo_B': equipo_b,
        'lambda_A': round(lam_a, 3), 'lambda_B': round(lam_b, 3),
        'P_Victoria_A': round(p_a, 4), 'P_Empate': round(p_e, 4), 'P_Victoria_B': round(p_b, 4),
        'Ganador_Pred': ganador,
    }

# Bracket real y conocido de dieciseisavos de final (32avos -> 16)
BRACKET_R32 = [
    ('Sudáfrica', 'Canadá'), ('Brasil', 'Japón'), ('Alemania', 'Paraguay'), ('Países Bajos', 'Marruecos'),
    ('Costa de Marfil', 'Noruega'), ('Francia', 'Suecia'), ('México', 'Ecuador'), ('Inglaterra', 'RD Congo'),
    ('Bélgica', 'Senegal'), ('Estados Unidos', 'Bosnia y Herzegovina'), ('España', 'Austria'), ('Portugal', 'Croacia'),
    ('Suiza', 'Argelia'), ('Australia', 'Egipto'), ('Argentina', 'Cabo Verde'), ('Colombia', 'Ghana'),
]

pred_r32 = pd.DataFrame([analizar_partido_bayes(a, b, posteriors) for a, b in BRACKET_R32])
pred_r32

,Equipo_A,Equipo_B,lambda_A,lambda_B,P_Victoria_A,P_Empate,P_Victoria_B,Ganador_Pred
0,Sudáfrica,Canadá,1.024,1.052,0.3420,0.3016,0.3563,Canadá
1,Brasil,Japón,1.047,0.901,0.3825,0.3123,0.3051,Brasil
2,Alemania,Paraguay,1.133,1.094,0.3653,0.2892,0.3455,Alemania
3,Países Bajos,Marruecos,1.061,1.113,0.3401,0.2932,0.3666,Marruecos
4,Costa de Marfil,Noruega,1.314,0.980,0.4426,0.2790,0.2783,Costa de Marfil
5,Francia,Suecia,1.349,0.976,0.4534,0.2756,0.2709,Francia
6,México,Ecuador,0.972,0.815,0.3784,0.3289,0.2927,México
7,Inglaterra,RD Congo,1.042,0.963,0.3669,0.3077,0.3254,Inglaterra
8,Bélgica,Senegal,1.255,0.980,0.4259,0.2850,0.2890,Bélgica
9,Estados Unidos,Bosnia y Herzegovina,1.266,1.108,0.3997,0.2773,0.3229,Estados Unidos


## 9. Demostración: la actualización bayesiana al sumar un partido de octavos

Esto es lo que se pidió validar explícitamente: que al jugarse un partido nuevo (p. ej. de octavos), el resultado se suma a $\alpha$ y $\beta$, y la tasa de gol posterior cambia — sin reentrenar nada ni recalcular el hiperprior.

Tomamos un ejemplo **hipotético/simulado** (NO un resultado real): supongamos que México le gana a Ecuador 2–1 en dieciseisavos. Comparamos la posterior de ataque de México *antes* y *después* de sumar ese partido.

In [6]:
# --- Ejemplo SIMULADO, solo para demostrar el mecanismo de actualización ---
equipo_demo = 'México'
goles_nuevos_demo = 2  # goles a favor en el partido simulado

fila = agg.loc[equipo_demo]
ataque_antes = (a0_ataque + fila['goles_favor']) / (b0_ataque + fila['partidos'])

goles_favor_despues = fila['goles_favor'] + goles_nuevos_demo
partidos_despues = fila['partidos'] + 1
ataque_despues = (a0_ataque + goles_favor_despues) / (b0_ataque + partidos_despues)

print(f'{equipo_demo} ANTES  -> partidos={fila["partidos"]}, goles_favor={fila["goles_favor"]}, E[ataque]={ataque_antes:.4f}')
print(f'{equipo_demo} DESPUÉS -> partidos={partidos_despues}, goles_favor={goles_favor_despues}, E[ataque]={ataque_despues:.4f}')
print(f'Cambio en la tasa posterior de ataque: {ataque_despues - ataque_antes:+.4f}')

México ANTES  -> partidos=3, goles_favor=6, E[ataque]=1.8985
México DESPUÉS -> partidos=4, goles_favor=8, E[ataque]=1.8993
Cambio en la tasa posterior de ataque: +0.0008


Esto confirma el mecanismo: **cada partido nuevo desplaza la posterior**, y mientras menos partidos acumulados tenga el equipo, mayor es el desplazamiento relativo (la evidencia nueva pesa más cuando hay menos historia previa). Esta misma operación — sumar goles a $\alpha$, sumar 1 partido a $\beta$ — es la que ejecuta la app cada vez que se registra un resultado real de eliminación directa.

## 10. Congelar el hiperprior para la app

El hiperprior ($\alpha_0,\beta_0,\gamma_0,\delta_0,\mu_{\text{global}}$) se calculó **una sola vez**, con la fase de grupos completa (la muestra de referencia más grande y estable que vamos a tener). A partir de aquí queda **congelado**: la app de Streamlit no lo vuelve a recalcular cada vez que se registra un resultado de octavos/cuartos/etc. — solo recalcula la posterior de los equipos involucrados. Esto evita que la población de referencia se vaya angostando (y degradando) a medida que el torneo elimina equipos.

In [7]:
hiperprior = {
    'a0_ataque': a0_ataque, 'b0_ataque': b0_ataque,
    'a0_defensa': a0_defensa, 'b0_defensa': b0_defensa,
    'mu_global': MU_GLOBAL,
    'gol_max': GOL_MAX,
    'n_equipos_base': int(agg.shape[0]),
    'fuente': 'resultados_reales.csv (fase de grupos, 32 clasificados a dieciseisavos)',
    'calculado_en': datetime.now(timezone.utc).isoformat(),
}

with open('hiperprior_bayesiano.json', 'w', encoding='utf-8') as f:
    json.dump(hiperprior, f, ensure_ascii=False, indent=2)

print('hiperprior_bayesiano.json escrito:')
print(json.dumps(hiperprior, ensure_ascii=False, indent=2))

hiperprior_bayesiano.json escrito:
{
  "a0_ataque": 219.78681506849364,
  "b0_ataque": 115.93150684931531,
  "a0_defensa": 11.70878998815165,
  "b0_defensa": 11.129146919431273,
  "mu_global": 1.8958333333333335,
  "gol_max": 7,
  "n_equipos_base": 32,
  "fuente": "resultados_reales.csv (fase de grupos, 32 clasificados a dieciseisavos)",
  "calculado_en": "2026-06-28T18:50:02.812877+00:00"
}


---
**Elaborado con modelo Poisson-Gamma bayesiano**  
Prior: Gamma($\alpha_0,\beta_0$) estimado por Empirical Bayes (método de momentos) sobre 32 equipos clasificados.  
Actualización conjugada: $\alpha_{\text{nuevo}} = \alpha_0 + \sum \text{goles}$, $\beta_{\text{nuevo}} = \beta_0 + n_{\text{partidos}}$.  
$\lambda_{i\to j} = \mathbb{E}[\text{ataque}_i]\times\mathbb{E}[\text{defensa}_j] / \mu_{\text{global}}$  
Salida: `hiperprior_bayesiano.json` (consumido por `app.py` junto con `resultados_reales.csv` y `resultados_eliminacion.json`).